# StockSense2.0: real sales to Power BI decisions

Main experiments use UCI Online Retail II, Chen,D.(2012), https://doi.org/10.24432/C5CG6D, CC BY4.0. StockSense filters/aggregates this source. Profits are modeled proxies with assumed costs; sales are not necessarily true demand. Install the project first; Power BI is the presentation layer.


In [ ]:
from pathlib import Path
import sys,json
root=Path.cwd() if (Path.cwd()/"stocklab").exists() else Path.cwd().parent
sys.path.insert(0,str(root))
import numpy as np
from dataclasses import replace
from stocklab.retail import catalog,product_data,retail_run,backtest_all
from stocklab.model import Scenario,uniforms,worlds,outcomes,analytic_order,analytic_expected_profit


## 1. Randomness has a replay button
A seed reproduces the random experiment; repeating it does not create independent evidence.


In [ ]:
points=np.random.default_rng(42).random((100000,2))
print("Pi estimate:",4*np.mean(np.sum(points**2,axis=1)<=1))


## 2. Audit the actual data
Every exclusion is counted once. Customer IDs are unnecessary for weekly SKU aggregation and absent from exports.


In [ ]:
m=catalog()
assert m["raw_rows"]-m["cleaned_rows"]==sum(a["removed"] for a in m["cleaning_audit"])
print(m["citation"],m["license"])
print(m["cleaning_audit"])
_,product,rows,values=product_data()
assert len(values)==104
print(product,"Zero sales weeks:",int(sum(values==0)))


## 3. Preserve observed zeros
Analogy: sample cards representing observed weeks. Drawing more cards reduces numerical error but adds no new real observations.


In [ ]:
history=np.array([0,0,1,2,3,4,5,9])
d,_=worlds(Scenario(),uniforms(8192,"sobol",20),history)
assert set(d)==set(history) and abs(np.mean(d==0)-.25)<.002
print("Empirical zero frequency:",np.mean(d==0))


## 4. Select and independently evaluate a real-sales policy
The final-refit simulation plans the archival week2011-12-05. Its numerical CI is conditional on the historical distribution and assumed economics.


In [ ]:
result=retail_run(n=4096,replicates=12,seed=2026)
assert result["config"]["demand_model"]=="empirical_weekly_sales"
print(result["config"]["data_context"])
print(result["selected"])
print(result["paired_delta_ci95"])


## 5. Preferences affect service as well as profit
A higher score does not guarantee a minimum service level.


In [ ]:
for w in [0,.2,.5]:
 r=retail_run(overrides={"risk_weight":w},n=1024,replicates=4)
 print(w,r["selected"]["order"],r["selected"]["mean_profit"],r["selected"]["fill_rate"])


## 6. Inspect all products and all four historical policies
At each test week use only the preceding52 observations.13 validation weeks provide diagnostics;39 test weeks provide sequential replay. Exact empirical scoring avoids simulation noise in choosing these backtest policies.


In [ ]:
replay=backtest_all()
for p in replay["products"]:
 print("\n",p["stock_code"],p["description"])
 for policy in p["policies"]:
  print(policy["policy"],round(policy["mean_profit_proxy_gbp"],2),round(policy["fill_rate_proxy"],3))
 assert len(p["weekly_records"])==39*4
 assert all(r["training_last_week"]<r["week_start"] for r in p["weekly_records"])


## 7. Separate synthetic accounting and analytic references
These illustrative numbers verify mathematics and are not UCI financial measurements.


In [ ]:
s=Scenario()
profit,_,_=outcomes(s,100,np.array([80.,120.]),np.ones(2))
np.testing.assert_array_equal(profit,[1360,2100])
simple=replace(s,disruption_probability=0,risk_weight=0)
q=analytic_order(simple);truth=analytic_expected_profit(simple,q)
d,f=worlds(simple,uniforms(65536,"sobol",17))
print("Analytic:",truth,"Sobol:",outcomes(simple,q,d,f)[0].mean())


## 8. Inspect the Power BI mart
Python computes the scenarios; Power Query imports; DAX applies filter context; visuals explain. Slicers never trigger Python. Open the native project using powerbi/README.md.


In [ ]:
mart=json.loads((root/"powerbi/data/export_manifest.json").read_text())
assert mart["product_scenario_count"]==135
print(mart["table_rows"])
print("Fresh simulations: python scripts/export_powerbi.py, then Refresh Power BI")


## 9. Challenge the recommendation
The default risk-sensitive policy beats trailing-mean modeled profit on four products but loses on jumbo bags and lowers fill on all five. Explain what inventory/cost/service data would change your recommendation. Extend with a declared fill-rate constraint and historical-only inputs; test feasible and infeasible cases. See the44-lesson study guide and Power BI Windows acceptance checklist.
